# Future Plans

Ideas for improving Tigers and Goats training. This notebook describes future work; it does not run training or enable these features.

**Status:** Proposed | **Added:** 2026-10-19

All configuration examples below are **proposed, not currently supported**. They show how a future interface could look. Do not paste them into the current training configuration.

## Complexity at a glance

Complexity describes development and testing effort, not model strength.

| Idea | Complexity | Main work |
| --- | --- | --- |
| 1. Multiple probabilistically mixed opponents | High | Select from a model pool, validate settings, load opponents and support evaluation |
| 2. Sticky Markov opponent sampling | Medium after Plan 1 | Remember the last opponent and control when to sample again |
| 3. Direct randomized starting states | High | Generate playable states and initialize all game bookkeeping correctly |

Keep each idea short. Record completed work and experiment results in [Versions and progress](VERSIONS_AND_PROGRESS.ipynb).

## 1. Multiple probabilistically mixed opponents

**Goal:** Train one learner against a variety of scripted and saved opponents, rather than only two opponent types.

**Complexity: High.** This affects configuration, model loading and memory use, opponent selection, logging, and evaluation in both the notebook and training script.

### Proposed settings — not currently supported

Example for a goat learner; the saved file must contain a compatible Tiger policy:

```python
"opponent": ["greedy", "smart", "models/tiger_L2.zip"],
"mix_prob": [0.2, 0.3, 0.5],
```

The lists match by position: 20% Greedy Tiger, 30% Smart Tiger, and 50% the saved Tiger model. At the start of each game, select one opponent and keep it for the whole game. Actual proportions will vary, especially in short runs.

### Implementation needs

- Require nonempty lists of the same length.
- Require finite probabilities from 0 to 1. Their sum must equal 1 within a small floating-point tolerance; report invalid inputs clearly.
- Check scripted names for the learner's role: goats face Tiger opponents; tigers face Goat opponents. Check saved model files and compatibility before training.
- Keep opponent weights frozen. Load and reuse models efficiently, including within each training worker.
- Make sampling reproducible with a seed and preserve current string opponents and scalar mixing settings.
- Record opponent identities, requested probabilities, and observed selection frequencies.
- Provide matching training and evaluation support in the notebook and script. Evaluate against individual opponents as well as the mixture.

**Dependency:** Goats facing saved Tiger models need the loading and validation support described in the [Model Tiger training plan](../docs/MODEL_TIGER_TRAINING_PLAN.md). The current notebook already supports Tiger learners facing a saved Goat model, but not an arbitrary model pool.

### Success checks

Invalid list sizes, probabilities, names, and model files fail clearly. Every supported opponent works without changing its weights. Repeating a seed repeats the selection sequence. Across many games, observed selections approach the requested distribution. Existing configurations still work.

## 2. Sticky Markov opponent sampling

**Goal:** Let the learner play several consecutive games against the same opponent before drawing again from the pool.

**Complexity: Medium after Plan 1.** This adds selection history, seeded sampling, and logging on top of the opponent pool.

### Proposed setting — not currently supported

Add this alongside the opponent and probability lists from Plan 1:

```python
"stickiness": 0.8,
```

### Behavior

- The first game samples an opponent using the pool's probability list.
- Before each later game, keep the previous opponent with 80% probability. Otherwise, sample from the pool again.
- A new draw can select the same opponent, so 80% stickiness does not mean exactly a 20% chance of changing opponents.
- Change opponents only between games. Each environment keeps its own previous opponent.
- Start a fresh selection history at each curriculum phase.
- `stickiness = 0` means independent sampling each game. `stickiness = 1` keeps the initial opponent for the whole phase.

Validate stickiness as a finite number from 0 to 1. This controls the chance of staying, not a fixed number of games or training steps. The pool's probabilities describe game selections, not exact shares of training steps; longer games contribute more steps. With stickiness of 1, a single environment never explores the rest of its pool during that phase.

**Learning question:** Does spending longer with each opponent help adaptation, or does it encourage forgetting other opponents? Compare it against independent sampling with the same pool and training budget.

This changes how experience is collected. PPO uses a rollout buffer; this idea does **not** create a replay buffer or guarantee better learning.

### Success checks

Test stickiness of 0 and 1, invalid values, reproducible selections, separate histories across environments, and resets at phase boundaries. Log opponent streaks and observed frequencies. Verify that an opponent never changes mid-game.

## 3. Direct randomized starting states

**Goal:** Expose PPO to positions it rarely visits by sampling starting states directly, rather than only playing random warm-up moves from the normal opening.

**Complexity: High.** A valid state includes more than the visible board. Game counters, legal actions, and reward bookkeeping must agree with the position.

### Proposed setting — not currently supported

```python
"random_start_prob": 0.5,
```

At each episode reset, this means approximately 50% normal openings and 50% directly sampled starting states. Validate the setting as a finite probability from 0 to 1. A value of 0 preserves normal starts; 1 uses randomized starts for every training episode.

### Implementation needs

- Sample positions in both placement and movement phases, including piece locations and capture counts.
- Enforce game constraints: piece counts, phase, captures, and placement progress must agree.
- Reject invalid positions and positions where the game is already over.
- Initialize legal moves, turn counters, repetition tracking, and reward bookkeeping consistently with the starting scenario. Define the starting-history rules before implementation so the first reward and timeout behavior are meaningful.
- Use seeded randomness and bounded retries when rejecting candidates. Report failure clearly if no valid position can be generated within the retry limit.
- Support both learner roles and the selected scripted or model opponent.
- Save random-start settings and observed start statistics, including normal/random proportions and placement/movement proportions.

Positions can be unusual and do not need a demonstrated sequence of moves from the opening. However, passing basic validity checks does not prove a position is reachable in ordinary play. The goal is broader coverage of valid scenarios, not a claim that every generated position occurs in a normal game or that every state will be visited.

### Success checks and experiment

Check observations, piece counts, legal-action masks, and first-step reward bookkeeping. Test reproducibility, rejected candidates, retry limits, both learner roles, and both game phases. Confirm that `random_start_prob = 0` leaves normal reset behavior unchanged.

Compare normal-only training with mixed-start training using equal training-step budgets and multiple seeds. Report performance separately from normal openings and from held-out randomized positions. Also report generation overhead. Broader state coverage is useful only if the measured learning results support it.

## Related plans and references

- [Model Tiger training plan](../docs/MODEL_TIGER_TRAINING_PLAN.md): support for Goat learners facing frozen saved Tiger policies.
- [Network architecture and model combiner](../docs/NETWORK_ARCHITECTURE_AND_MODEL_COMBINER.md): current policy structure and an idea for combining trained models.
- [Versions and progress](VERSIONS_AND_PROGRESS.ipynb): completed changes, results, and next steps.

These are ideas and supporting references, not promises that the features already exist.

## Template for a new idea

Copy this into a new Markdown cell above the template:

### [Idea title]

**Status:** Proposed  
**Goal:** [What should improve, and for whom?]  
**Complexity:** Low / Medium / High — [Why?]

**Proposed settings — not currently supported:** [Optional example.]

**Behavior and implementation needs:** [How it should work and what must change.]

**Dependencies:** [Earlier features or documents needed first.]

**Success checks:** [How to verify correct behavior and measure whether the idea helps.]